# Aggregate Parallel Model Results

Attach all seven per-model Kaggle output datasets. This notebook merges `eval_multifold/` and `xai_multifold/`, creates 5-fold tables, renders figures, and runs the rethought feature-map coherence metric.

In [ ]:
%pip install -q timm pydicom captum grad-cam scikit-image scipy seaborn

from pathlib import Path
import json, os, shutil, subprocess, sys
import pandas as pd

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
PY = sys.executable

MODELS = ["convnext_blackbox", "cbm_nonleaky", "cbm_leaky", "resnet50", "densenet121", "efficientnet_b4", "vit_small"]
THEORY_MODELS = ["resnet50", "densenet121", "convnext_blackbox", "efficientnet_b4", "vit_small"]

CFG_NAME = {
    "convnext_blackbox": "configs/baselines/convnext_blackbox.yaml",
    "cbm_nonleaky": "configs/baselines/cbm_nonleaky.yaml",
    "cbm_leaky": "configs/baselines/cbm_leaky.yaml",
    "resnet50": "configs/baselines/resnet50.yaml",
    "densenet121": "configs/baselines/densenet121.yaml",
    "efficientnet_b4": "configs/baselines/efficientnet_b4.yaml",
    "vit_small": "configs/baselines/vit_small.yaml",
}
EXP = {
    "convnext_blackbox": "baseline_convnext_tiny_blackbox",
    "cbm_nonleaky": "cbm_nonleaky_7concepts",
    "cbm_leaky": "cbm_leaky_5concepts",
    "resnet50": "baseline_resnet50",
    "densenet121": "baseline_densenet121",
    "efficientnet_b4": "baseline_efficientnet_b4",
    "vit_small": "baseline_vit_small",
}

def run(cmd):
    print("\n$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), check=True)

def input_roots():
    roots = [WORK]
    if INPUT.exists():
        roots += [p for p in INPUT.iterdir() if p.is_dir()]
        datasets = INPUT / "datasets"
        if datasets.exists():
            for owner in datasets.iterdir():
                if owner.is_dir():
                    roots += [p for p in owner.iterdir() if p.is_dir()]
    return roots

def looks_like_code(p):
    return (
        (p / "scripts" / "train.py").exists()
        and (p / "configs").exists()
        and (p / "scripts" / "feature_map_smoothness.py").exists()
    )

def first_existing(candidates, label):
    for p in map(Path, candidates):
        if p.exists():
            return p
    preview = [str(p) for p in candidates[:10]]
    raise FileNotFoundError(f"Could not find {label}. Tried: {preview}")

def find_code_source():
    candidates = [INPUT / "spinexnet-code", INPUT / "spinexnet-code" / "het-spine"]
    for r in input_roots():
        candidates += [r, r / "het-spine", r / "spinexnet-code"]
    for p in candidates:
        if looks_like_code(p):
            return p
    raise FileNotFoundError("Could not find spinexnet-code")

def find_manifest():
    candidates = []
    for r in input_roots():
        candidates += [r / "manifest_v2.csv", r / "manifests" / "manifest_v2.csv"]
    return first_existing(candidates, "manifest_v2.csv")

def find_cache():
    candidates = []
    for r in input_roots():
        candidates += [r / "image_cache_224", r / "pre-processed-crop-224" / "image_cache_224"]
    candidates = [p for p in candidates if (p / "images_uint8.npy").exists()]
    return first_existing(candidates, "image_cache_224")

SRC = find_code_source()
CODE = WORK / "spinexnet-code"
if SRC.resolve() != CODE.resolve():
    shutil.copytree(SRC, CODE, dirs_exist_ok=True)
MANIFEST = find_manifest()
CACHE = find_cache()
CFG = {m: CODE / rel for m, rel in CFG_NAME.items()}

def find_checkpoint(model, fold):
    rels = [
        Path("outputs") / EXP[model] / f"fold_{fold}" / "best.pt",
        Path(EXP[model]) / f"fold_{fold}" / "best.pt",
        Path("checkpoints") / f"{model}_fold_{fold}_best.pt",
        Path("checkpoints") / f"{model}_fold{fold}_best.pt",
    ]
    if fold == 0:
        rels.append(Path("checkpoints") / f"{model}_best.pt")
    candidates = []
    for r in input_roots():
        candidates += [r / rel for rel in rels]
    candidates += [WORK / rel for rel in rels]
    return first_existing(candidates, f"{model} fold {fold} checkpoint")

def merge_tree_named(name):
    dest = WORK / f"combined_{name}"
    for r in input_roots():
        src = r / name
        if src.exists():
            try:
                if src.resolve() == dest.resolve():
                    continue
            except Exception:
                pass
            shutil.copytree(src, dest, dirs_exist_ok=True)
    return dest

EVAL_ROOT = merge_tree_named("eval_multifold")
XAI_ROOT = merge_tree_named("xai_multifold")
print("CODE:", CODE)
print("MANIFEST:", MANIFEST)
print("CACHE:", CACHE)
print("EVAL_ROOT:", EVAL_ROOT)
print("XAI_ROOT:", XAI_ROOT)


In [ ]:

rows = []
for path in EVAL_ROOT.glob("*/fold_*/metrics_val.json"):
    with open(path) as f:
        d = json.load(f)
    rows.append({"model": path.parents[1].name, "fold": path.parent.name, **d})

classification = pd.DataFrame(rows)
if classification.empty:
    print("No classification metrics found.")
else:
    classification["fold_idx"] = classification["fold"].str.extract(r"(\d+)").astype(int)
    classification = classification.sort_values(["model", "fold_idx"]).drop(columns=["fold_idx"])
    classification.to_csv(WORK / "classification_metrics_by_fold.csv", index=False)
    metrics = ["weighted_log_loss", "balanced_accuracy", "macro_f1", "accuracy", "auc_ovr"]
    summary = classification.groupby("model")[metrics].agg(["mean", "std"]).round(4)
    summary.to_csv(WORK / "classification_metrics_mean_std.csv")
    display(summary)


In [ ]:

rows = []
for path in XAI_ROOT.glob("fold_*/*/xai_summary_v2.json"):
    with open(path) as f:
        d = json.load(f)["summary"]
    rows.append({
        "fold": path.parents[1].name,
        "model": path.parent.name,
        "mean_spearman": d.get("mean_spearman"),
        "mean_top20_iou": d.get("mean_top20_iou"),
        "consensus_insertion_auc_mean": d.get("consensus_insertion_auc_mean"),
        "consensus_expert_roi_mean": d.get("consensus_expert_roi_mean"),
    })

xai_summary = pd.DataFrame(rows)
if xai_summary.empty:
    print("No XAI summaries found.")
else:
    xai_summary["fold_idx"] = xai_summary["fold"].str.extract(r"(\d+)").astype(int)
    xai_summary = xai_summary.sort_values(["model", "fold_idx"]).drop(columns=["fold_idx"])
    xai_summary.to_csv(WORK / "xai_multifold_summary_by_fold.csv", index=False)
    xai_mean_std = xai_summary.groupby("model")[[
        "mean_spearman",
        "mean_top20_iou",
        "consensus_insertion_auc_mean",
        "consensus_expert_roi_mean",
    ]].agg(["mean", "std"]).round(4)
    xai_mean_std.to_csv(WORK / "xai_multifold_summary_mean_std.csv")
    display(xai_mean_std)


In [ ]:

FIG = WORK / "figures"
XAI_FOLD0 = XAI_ROOT / "fold_0"

for model in MODELS:
    if not (XAI_FOLD0 / model / "xai_summary_v2.json").exists():
        print("Missing XAI fold_0 for", model)
        continue
    run([PY, CODE / "scripts/visualize_xai.py", "--results-dir", XAI_FOLD0 / model, "--output-dir", FIG / "fold_0" / model])

run([PY, CODE / "scripts/visualize_xai.py", "--cross-model-dir", XAI_FOLD0, "--output-dir", FIG / "fold_0" / "cross_model"])
run([PY, CODE / "scripts/generate_gallery.py", "--xai-dir", XAI_FOLD0, "--models", "densenet121", "convnext_blackbox", "vit_small", "--manifest", MANIFEST, "--cache-dir", CACHE, "--output-dir", FIG / "fold_0" / "gallery"])


In [ ]:

run([
    PY, CODE / "scripts/feature_map_smoothness.py",
    "--configs", *[CFG[m] for m in THEORY_MODELS],
    "--checkpoints", *[find_checkpoint(m, 0) for m in THEORY_MODELS],
    "--names", *THEORY_MODELS,
    "--manifest", MANIFEST,
    "--fold", 0,
    "--cache-dir", CACHE,
    "--output-dir", WORK / "feature_map_smoothness",
    "--max-samples", 300,
    "--agreement-csv", FIG / "fold_0" / "cross_model" / "cross_model_agreement.csv",
])


In [ ]:

for model in ["convnext_blackbox", "vit_small"]:
    run([
        PY, CODE / "scripts/model_randomization.py",
        "--config", CFG[model],
        "--checkpoint", find_checkpoint(model, 0),
        "--manifest", MANIFEST,
        "--fold", 0,
        "--cache-dir", CACHE,
        "--output-dir", WORK / "randomization" / model,
        "--methods", "gradcam", "integrated_gradients", "gradient_shap", "occlusion",
        "--max-samples", 50,
        "--n-levels", 5,
    ])

for model in ["cbm_nonleaky", "cbm_leaky"]:
    run([
        PY, CODE / "scripts/concept_intervention.py",
        "--config", CFG[model],
        "--checkpoint", find_checkpoint(model, 0),
        "--manifest", MANIFEST,
        "--fold", 0,
        "--cache-dir", CACHE,
        "--output-dir", WORK / "intervention" / model,
        "--max-samples", 1000,
    ])
